# Airport PageRank Project

## Using PageRank to identify important airports in the U.S. flight network

This notebook applies the PageRank algorithm to a directed U.S. airport route network. Airports are represented as nodes, and direct flight routes are represented as directed edges from an origin airport to a destination airport.

The goal is to measure airport importance using network structure, not just the number of direct routes. An airport receives a higher PageRank score when it is connected to other airports that are themselves important in the network.

In this notebook, we will:

- Load OpenFlights airport and route data.
- Filter the data to direct routes between U.S. airports.
- Build an adjacency matrix and transition matrix.
- Compute PageRank with power iteration.
- Rank airports by PageRank score.
- Simulate the closure of a major hub and compare how rankings change.


## 1. Import Libraries

We begin by importing the packages used for data handling, numerical computation, and visualization. The analysis mainly relies on `pandas` for tabular data, `numpy` for matrix operations, and `matplotlib` for plotting.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)


## 2. Load OpenFlights Data

The project uses two OpenFlights data files:

- `airports.dat` contains airport metadata, including airport name, city, country, IATA code, latitude, and longitude.
- `routes.dat` contains airline route records, including source and destination airports.

The route data is directional: a route from airport A to airport B is treated separately from a route from airport B to airport A. This matters because PageRank is computed on a directed graph.


In [25]:
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

if not DATA_DIR.exists():
    raise FileNotFoundError(
        "Expected a data/ folder next to this notebook. "
        "Run the notebook from the airport_pagerank_project directory."
    )

airport_columns = [
    "airport_id", "name", "city", "country", "iata", "icao",
    "latitude", "longitude", "altitude", "timezone", "dst",
    "tz_database_time_zone", "type", "source",
]

route_columns = [
    "airline", "airline_id", "source_airport", "source_airport_id",
    "destination_airport", "destination_airport_id", "codeshare",
    "stops", "equipment",
]

airports = pd.read_csv(
    DATA_DIR / "airports.dat",
    names=airport_columns,
    na_values="\\N",
    keep_default_na=False,
)
routes = pd.read_csv(
    DATA_DIR / "routes.dat",
    names=route_columns,
    na_values="\\N",
    keep_default_na=False,
)

airports.head()


,airport_id,name,city,country,iata,icao,latitude,longitude,altitude,timezone,dst,tz_database_time_zone,type,source
0,1,Goroka Airport,Goroka,Papua New Guinea,GKA,AYGA,-6.081690,145.391998,5282,10.0,U,Pacific/Port_Moresby,airport,OurAirports
1,2,Madang Airport,Madang,Papua New Guinea,MAG,AYMD,-5.207080,145.789001,20,10.0,U,Pacific/Port_Moresby,airport,OurAirports
2,3,Mount Hagen Kagamuga Airport,Mount Hagen,Papua New Guinea,HGU,AYMH,-5.826790,144.296005,5388,10.0,U,Pacific/Port_Moresby,airport,OurAirports
3,4,Nadzab Airport,Nadzab,Papua New Guinea,LAE,AYNZ,-6.569803,146.725977,239,10.0,U,Pacific/Port_Moresby,airport,OurAirports
4,5,Port Moresby Jacksons International Airport,Port Moresby,Papua New Guinea,POM,AYPY,-9.443380,147.220001,146,10.0,U,Pacific/Port_Moresby,airport,OurAirports


## 3. Filter to the U.S. Route Network

Next, we restrict the data to U.S. airports with valid IATA codes. We then keep only direct routes where both the source and destination airports are in the U.S. network.

The OpenFlights route file can contain multiple airline-level records for the same airport pair. We aggregate those duplicates so each directed airport pair appears once in the route table. For the adjacency matrix, each existing directed route is represented with a `1`, and each missing route is represented with a `0`.


In [24]:
us_airports = airports[
    (airports["country"] == "United States")
    & (airports["iata"].notna())
    & (airports["iata"] != "")
    & (airports["type"] == "airport")
].copy()

# Use one metadata row per IATA code so matrix rows map cleanly to airports.
us_airports = us_airports.drop_duplicates(subset="iata", keep="first")
us_iata_codes = set(us_airports["iata"])

us_routes = routes[
    (routes["source_airport"].isin(us_iata_codes))
    & (routes["destination_airport"].isin(us_iata_codes))
    & (routes["stops"] == 0)
].copy()

route_edges = (
    us_routes.groupby(["source_airport", "destination_airport"])
    .size()
    .reset_index(name="route_count")
)

active_airports = sorted(
    set(route_edges["source_airport"]).union(route_edges["destination_airport"])
)

airport_lookup = (
    us_airports.set_index("iata")
    .loc[active_airports, ["name", "city", "country", "latitude", "longitude"]]
    .reset_index()
    .rename(columns={"index": "iata"})
)

print(f"Airports in network: {len(airport_lookup):,}")
print(f"Directed route edges: {len(route_edges):,}")
route_edges.head()


Airports in network: 549
Directed route edges: 5,450


,source_airport,destination_airport,route_count
0,ABE,ATL,3
1,ABE,CLT,2
2,ABE,DTW,1
3,ABE,MYR,1
4,ABE,ORD,1


## 4. Build the Adjacency Matrix

We now convert the route table into a binary adjacency matrix. The notebook uses the standard PageRank matrix convention.


$$
A_{ij} =
\begin{cases}
1, & \text{if there is a route from airport } j \text{ to airport } i \\
0, & \text{otherwise}
\end{cases}
$$


Under this convention, each column represents a departure airport and each row represents an arrival airport. For example, if there is a route from `ATL` to `LAX`, the value is stored as `1` in the row for `LAX` and the column for `ATL`.

Using this orientation makes the later transition matrix column-stochastic, meaning each column can be interpreted as a probability distribution over possible next airports.


In [26]:
def build_adjacency_matrix(airport_codes, route_edges):
    """Build a binary matrix where A[i, j] = 1 if j has a route to i."""
    code_to_index = {code: i for i, code in enumerate(airport_codes)}
    n = len(airport_codes)
    adjacency = np.zeros((n, n), dtype=int)

    for _, row in route_edges.iterrows():
        source = row["source_airport"]
        destination = row["destination_airport"]
        adjacency[code_to_index[destination], code_to_index[source]] = 1

    return adjacency


airport_codes = list(airport_lookup["iata"])
n = len(airport_codes)
A = build_adjacency_matrix(airport_codes, route_edges)

A.shape


(549, 549)

### View the Full Adjacency Matrix

The table below shows the full labeled adjacency matrix. Rows represent destination airports, columns represent source airports, and each value is either `0` or `1`:

- `1` means there is a direct route from the column airport to the row airport.
- `0` means there is no direct route from the column airport to the row airport.

Because the full matrix is large, the notebook also saves it to `outputs/adjacency_matrix.csv` for easier inspection in a spreadsheet or text editor.


In [ ]:
adjacency_matrix = pd.DataFrame(A, index=airport_codes, columns=airport_codes)
adjacency_matrix.to_csv(OUTPUT_DIR / "adjacency_matrix.csv")

pd.set_option("display.max_rows", len(adjacency_matrix))
pd.set_option("display.max_columns", len(adjacency_matrix.columns))
pd.set_option("display.width", None)

adjacency_matrix


## 5. Create the Transition Matrix

PageRank treats movement through the airport network as a probability process. The adjacency matrix tells us whether a route exists, but the transition matrix tells us the probability of moving from one airport to another.

For each departure airport `j`, we normalize the corresponding column of the adjacency matrix.


$$
M_{ij} = \frac{A_{ij}}{\sum_i A_{ij}}
$$


where:

- $A_{ij}$ is 1 if there is a direct route from airport $j$ to airport $i$, and 0 otherwise.
- $\sum_i A_{ij}$ is the total number of outgoing routes from airport $j$.
- $M_{ij}$ is the probability of moving from airport $j$ to airport $i$ by following one of its outgoing routes.

After normalization, each column of $M$ sums to 1.


$$
\sum_i M_{ij} = 1
$$


If an airport has no outgoing routes, it is called a dangling node. For a dangling column, we assign equal probability to every airport.


$$
M_{ij} = \frac{1}{N}
$$


where $N$ is the total number of airports in the network.


In [ ]:
def build_transition_matrix(A):
    """Convert an adjacency matrix into a column-stochastic transition matrix."""
    if A.ndim != 2 or A.shape[0] != A.shape[1]:
        raise ValueError("A must be a square matrix.")

    n = A.shape[0]
    if n == 0:
        raise ValueError("A must contain at least one airport.")

    M = A.astype(float).copy()
    column_sums = M.sum(axis=0)
    dangling_columns = column_sums == 0

    M[:, ~dangling_columns] /= column_sums[~dangling_columns]
    M[:, dangling_columns] = 1.0 / n

    return M


M = build_transition_matrix(A)
np.allclose(M.sum(axis=0), 1)


## 6. Run PageRank with Power Iteration

The transition matrix alone would rank airports based only on following routes. PageRank adds a small probability of random movement, called teleportation, so the model always converges and does not get stuck in one part of the network.

The Google Matrix is:


$$
G = dM + (1-d)\frac{1}{N}ee^T
$$


where:

- $G$ is the Google Matrix.
- $M$ is the transition matrix.
- $d$ is the damping factor. Here, $d = 0.85$.
- $N$ is the number of airports.
- $e$ is a vector of ones.

This means a traveler follows an actual route with probability $0.85$ and randomly jumps to any airport with probability $0.15$.

The PageRank vector is updated repeatedly using power iteration.


$$
v_{k+1} = Gv_k
$$


In the code, this is computed without explicitly storing the full Google Matrix.


$$
v_{k+1} = dMv_k + \frac{1-d}{N}e
$$


The algorithm stops when the L1 change between two consecutive vectors is smaller than the tolerance.


$$
\lVert v_{k+1} - v_k \rVert_1 < \epsilon
$$


The final vector $v$ contains one PageRank score for each airport. Larger scores indicate airports that are more central in the network.


In [ ]:
def pagerank_power_iteration(M, damping=0.85, tolerance=1e-10, max_iter=1_000):
    """Compute PageRank scores with power iteration."""
    if M.ndim != 2 or M.shape[0] != M.shape[1]:
        raise ValueError("M must be a square transition matrix.")
    if not 0 < damping < 1:
        raise ValueError("damping must be between 0 and 1.")

    n = M.shape[0]
    v = np.ones(n) / n
    delta = np.inf

    for iteration in range(1, max_iter + 1):
        next_v = damping * (M @ v) + (1 - damping) / n
        delta = np.linalg.norm(next_v - v, ord=1)
        v = next_v

        if delta < tolerance:
            return v, iteration, delta

    return v, max_iter, delta


scores, iterations, delta = pagerank_power_iteration(M)

print(f"Converged in {iterations} iterations")
print(f"Final L1 delta: {delta:.3e}")
print(f"PageRank scores sum to {scores.sum():.6f}")


## 7. Rank Airports by PageRank

The PageRank algorithm returns a score vector.


$$
v =
\begin{bmatrix}
v_1 \\
v_2 \\
\vdots \\
v_N
\end{bmatrix}
$$


where each entry $v_i$ is the PageRank score for airport $i$.

The scores form a probability distribution, so they sum to 1.


$$
\sum_i v_i = 1
$$


To make the vector interpretable, we join each score back to its airport metadata, including IATA code, airport name, and city. Airports are then sorted from highest to lowest PageRank.

A high score means an airport is central in the directed route network, especially if it receives routes from other airports that are also important.


In [ ]:
def rank_airports(airport_lookup, scores):
    rankings = airport_lookup.copy()
    rankings["pagerank"] = scores
    rankings = rankings.sort_values("pagerank", ascending=False).reset_index(drop=True)
    rankings["rank"] = rankings.index + 1
    return rankings[["rank", "iata", "name", "city", "pagerank", "latitude", "longitude"]]


rankings = rank_airports(airport_lookup, scores)
rankings[["rank", "iata", "name", "city", "pagerank"]].head(15)


## 8. Visualize the Top Airports

The top airports are selected using:


$$
\text{Top airports} = \operatorname{argmax}_{15}(v_i)
$$


This means we take the 15 airports with the largest PageRank scores.

The bar chart below visualizes those scores. The chart is sorted from smallest to largest within the top group so that the highest-ranked airports appear at the top of the final plot.

This visualization helps compare both rank order and relative score size. If one airport has a much longer bar, it is much more central according to the PageRank model.


In [ ]:
top = rankings.head(15).sort_values("pagerank", ascending=True)

plt.figure(figsize=(10, 6), dpi=150)
plt.barh(top["iata"], top["pagerank"], color="#20808D")
plt.title("Top U.S. airports by binary PageRank", fontweight="bold")
plt.xlabel("PageRank score")
plt.ylabel("Airport")
plt.grid(axis="x", alpha=0.25)
plt.gca().spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()


## 9. Simulate Closing a Major Hub

To study network resilience, we simulate the closure of a major hub airport. In this example, `ATL` is removed by deleting all routes into and out of that airport.

If `h` is the hub airport being removed, the perturbed route set is:


$$
E_{after} = \{(u, v) \in E : u \ne h \text{ and } v \ne h\}
$$


where:

- $E$ is the original set of directed routes.
- $u$ is a route source airport.
- $v$ is a route destination airport.
- $h$ is the removed hub airport.

After removing those routes, we rebuild the adjacency matrix, transition matrix, and PageRank vector.


$$
A_{after} \rightarrow M_{after} \rightarrow v_{after}
$$


This allows us to compare the baseline network against a disrupted network and identify which airports become more central after the hub closure.


In [ ]:
hub_to_remove = "ATL"

perturbed_edges = route_edges[
    (route_edges["source_airport"] != hub_to_remove)
    & (route_edges["destination_airport"] != hub_to_remove)
].copy()

A_after = build_adjacency_matrix(airport_codes, perturbed_edges)
M_after = build_transition_matrix(A_after)
scores_after, iterations_after, delta_after = pagerank_power_iteration(M_after)

after_rankings = rank_airports(airport_lookup, scores_after)

print(f"Removed hub: {hub_to_remove}")
print(f"Remaining directed route edges: {len(perturbed_edges):,}")
print(f"Post-closure convergence: {iterations_after} iterations")
after_rankings[["rank", "iata", "name", "city", "pagerank"]].head(15)


## 10. Compare Rankings Before and After Hub Closure

The comparison table joins the baseline PageRank results with the post-closure results. For each airport, we compare both rank and score.

The rank change is calculated as:


$$
\text{rank\_change}_i = \text{rank}_{before,i} - \text{rank}_{after,i}
$$


A positive value means the airport moved up in the ranking after the hub was removed. For example, moving from rank 20 to rank 12 gives:


$$
20 - 12 = 8
$$


The PageRank score change is calculated as:


$$
\text{pagerank\_change}_i = v_{after,i} - v_{before,i}
$$


A positive value means the airport gained PageRank score in the disrupted network.

The removed hub is excluded from the comparison so the table focuses on how the rest of the network responds.


In [ ]:
comparison = rankings[["iata", "name", "city", "rank", "pagerank"]].merge(
    after_rankings[["iata", "rank", "pagerank"]],
    on="iata",
    suffixes=("_before", "_after"),
)

comparison["rank_change"] = comparison["rank_before"] - comparison["rank_after"]
comparison["pagerank_change"] = comparison["pagerank_after"] - comparison["pagerank_before"]
comparison = comparison[comparison["iata"] != hub_to_remove].copy()

comparison.sort_values("rank_change", ascending=False).head(15)


## 11. Interpretation and Discussion

The baseline PageRank ranking identifies airports that are structurally important in the U.S. route network. These airports are not only well connected themselves, but also connected to other important airports.

In this notebook, the adjacency matrix is binary, so PageRank is based on whether a direct route exists rather than how many airlines fly that route or how many passengers use it.

The hub-closure experiment shows how centrality can shift when a major airport is removed. Airports with positive `rank_change` or `pagerank_change` become more important in the disrupted network because traffic paths and network influence are redistributed.

Key questions for the final report:

- Which airports have the highest baseline PageRank scores?
- Are the top PageRank airports also the airports with the most direct routes, or does PageRank reveal a different type of importance?
- What changes when `ATL` is removed from the network?
- Which airports gain PageRank after the closure, and why might those airports benefit?
- What does the result suggest about resilience and dependence on major hubs in the U.S. airport network?

Limitations:

- OpenFlights route data is historical, so the results should be interpreted as a demonstration of the method rather than a current operational ranking.
- The binary adjacency matrix treats every direct route equally, regardless of airline count, passenger volume, or flight frequency.
- The closure simulation removes routes in a simplified way and does not model real airline rerouting behavior.

Possible extensions:

- Compare PageRank with degree centrality or simple route counts.
- Repeat the closure simulation for other hubs such as `ORD`, `DFW`, `DEN`, or `LAX`.
- Build a weighted version using passenger volume or flight frequency.
- Add map-based visualizations using airport latitude and longitude.
